# 00b — Auditoria das Raws e Engenharia de Features

**Objetivo**: explorar as tabelas `raw_*` para validar a `trusted_municipios` e gerar a base enriquecida local com features derivadas.

**Inputs**: `trusted_municipios`, `raw_bcb_pix_transacoes`, `raw_bcb_estban`, `raw_anatel_banda_larga_fixa`.

**Outputs**:
- `data/processed/trusted_municipios_eda.parquet`
- `data/processed/reports/features_engineering_report.json`

In [1]:
# Imports
import logging

import numpy as np
import pandas as pd

from src.utils.bigquery import read_table_to_dataframe
from src.utils.eda import save_json, save_parquet

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# Leitura das tabelas
df = read_table_to_dataframe("trusted_municipios")
df_raw_pix = read_table_to_dataframe("raw_bcb_pix_transacoes")
df_raw_estban = read_table_to_dataframe("raw_bcb_estban")
df_raw_anatel = read_table_to_dataframe("raw_anatel_banda_larga_fixa")

logger.info("trusted: %s", df.shape)
logger.info("raw_pix: %s", df_raw_pix.shape)
logger.info("raw_estban: %s", df_raw_estban.shape)
logger.info("raw_anatel: %s", df_raw_anatel.shape)

2026-08-26 14:48:46,622 - INFO - trusted: (5570, 26)


2026-08-26 14:48:46,625 - INFO - raw_pix: (568236, 21)


2026-08-26 14:48:46,627 - INFO - raw_estban: (2914, 6)


2026-08-26 14:48:46,629 - INFO - raw_anatel: (5599, 9)


## 1. Inventário das colunas nas raws

In [3]:
inventory = {
    "trusted_municipios": list(df.columns),
    "raw_bcb_pix_transacoes": list(df_raw_pix.columns),
    "raw_bcb_estban": list(df_raw_estban.columns),
    "raw_anatel_banda_larga_fixa": list(df_raw_anatel.columns),
}
for table, cols in inventory.items():
    logger.info("%s: %d colunas", table, len(cols))

2026-08-26 14:48:46,784 - INFO - trusted_municipios: 26 colunas


2026-08-26 14:48:46,786 - INFO - raw_bcb_pix_transacoes: 21 colunas


2026-08-26 14:48:46,787 - INFO - raw_bcb_estban: 6 colunas


2026-08-26 14:48:46,788 - INFO - raw_anatel_banda_larga_fixa: 9 colunas


## 2. Auditoria Pix — soma dos meses vs trusted

A `trusted_municipios` utiliza `VL_PagadorPF` e `QT_PagadorPF` como proxy de volume e transações Pix.

In [4]:
pix_agg = (
    df_raw_pix.groupby("id_municipio")
    .agg(
        pix_total_volume_12m_raw=("VL_PagadorPF", "sum"),
        pix_total_transacoes_12m_raw=("QT_PagadorPF", "sum"),
    )
    .reset_index()
)

df_check = df[["id_municipio", "pix_total_volume_12m", "pix_total_transacoes_12m"]].merge(
    pix_agg, on="id_municipio", how="left"
)

diff_volume = (
    df_check["pix_total_volume_12m"] - df_check["pix_total_volume_12m_raw"].fillna(0)
).abs()
diff_trans = (
    df_check["pix_total_transacoes_12m"] - df_check["pix_total_transacoes_12m_raw"].fillna(0)
).abs()

logger.info("Diferença máxima volume: %.2f", diff_volume.max())
logger.info("Diferença máxima transações: %.2f", diff_trans.max())

2026-08-26 14:48:46,914 - INFO - Diferença máxima volume: 0.00


2026-08-26 14:48:46,917 - INFO - Diferença máxima transações: 0.00


## 3. Auditoria Estban — agências, depósitos e crédito

In [5]:
estban_agg = (
    df_raw_estban.groupby("id_municipio")
    .agg(
        quantidade_agencias_raw=("quantidade_agencias", "sum"),
        volume_depositos_raw=("volume_depositos", "sum"),
        volume_credito_raw=("volume_credito", "sum"),
    )
    .reset_index()
)

df_check_estban = df[
    ["id_municipio", "quantidade_agencias", "volume_depositos", "volume_credito"]
].merge(estban_agg, on="id_municipio", how="left")

for col in ["quantidade_agencias", "volume_depositos", "volume_credito"]:
    diff = (df_check_estban[col] - df_check_estban[f"{col}_raw"].fillna(0)).abs().max()
    logger.info("Diferença máxima %s: %.2f", col, diff)

2026-08-26 14:48:46,961 - INFO - Diferença máxima quantidade_agencias: 0.00


2026-08-26 14:48:46,965 - INFO - Diferença máxima volume_depositos: 0.00


2026-08-26 14:48:46,967 - INFO - Diferença máxima volume_credito: 0.00


## 4. Features derivadas

In [6]:
# Proporção PJ no volume Pix (pagador + recebedor)
pix_cols = ["VL_PagadorPF", "VL_PagadorPJ", "VL_RecebedorPF", "VL_RecebedorPJ"]
available_pix_cols = [c for c in pix_cols if c in df_raw_pix.columns]

if available_pix_cols:
    pix_side = (
        df_raw_pix.groupby("id_municipio")[available_pix_cols]
        .sum()
        .reset_index()
    )
    total_pix = pix_side[available_pix_cols].sum(axis=1)
    pj_cols = [c for c in available_pix_cols if "PJ" in c]
    pj_total = pix_side[pj_cols].sum(axis=1) if pj_cols else 0
    pix_side["pix_pj_pct"] = np.where(total_pix > 0, pj_total / total_pix, np.nan)
    df = df.merge(pix_side[["id_municipio", "pix_pj_pct"]], on="id_municipio", how="left")
else:
    df["pix_pj_pct"] = np.nan

# Ticket médio Pix
df["pix_ticket_medio"] = np.where(
    df["pix_total_transacoes_12m"].fillna(0) > 0,
    df["pix_total_volume_12m"] / df["pix_total_transacoes_12m"],
    np.nan,
)

# Flags e estratos
df["flag_sem_agencia"] = (df["quantidade_agencias"].fillna(0) == 0).astype(int)
df["estrato_populacional"] = pd.cut(
    df["populacao_total"],
    bins=[0, 50000, 500000, float("inf")],
    labels=["pequena", "media", "grande"],
)

# Features de eficiência bancária
df["depositos_por_agencia"] = np.where(
    df["quantidade_agencias"].fillna(0) > 0,
    df["volume_depositos"] / df["quantidade_agencias"],
    np.nan,
)
df["credito_por_agencia"] = np.where(
    df["quantidade_agencias"].fillna(0) > 0,
    df["volume_credito"] / df["quantidade_agencias"],
    np.nan,
)

logger.info("Features criadas. Shape final: %s", df.shape)

2026-08-26 14:48:47,146 - INFO - Features criadas. Shape final: (5570, 32)


## 5. Resumo das features

In [7]:
features_added = [
    "pix_pj_pct",
    "pix_ticket_medio",
    "flag_sem_agencia",
    "estrato_populacional",
    "depositos_por_agencia",
    "credito_por_agencia",
]

for feat in features_added:
    logger.info(
        "%s: nulos=%.2f%%", feat, df[feat].isnull().mean() * 100
    )

logger.info("Municípios sem agência: %d", int(df["flag_sem_agencia"].sum()))

2026-08-26 14:48:47,173 - INFO - pix_pj_pct: nulos=0.00%


2026-08-26 14:48:47,176 - INFO - pix_ticket_medio: nulos=0.00%


2026-08-26 14:48:47,179 - INFO - flag_sem_agencia: nulos=0.00%


2026-08-26 14:48:47,181 - INFO - estrato_populacional: nulos=0.00%


2026-08-26 14:48:47,183 - INFO - depositos_por_agencia: nulos=47.68%


2026-08-26 14:48:47,185 - INFO - credito_por_agencia: nulos=47.68%


2026-08-26 14:48:47,187 - INFO - Municípios sem agência: 2656


## 6. Salvamento

In [8]:
save_parquet(df, "trusted_municipios_eda.parquet")

features_report = {
    "features_adicionadas": features_added,
    "nulos_pix_pj_pct": float(df["pix_pj_pct"].isnull().mean() * 100),
    "nulos_pix_ticket_medio": float(df["pix_ticket_medio"].isnull().mean() * 100),
    "municipios_sem_agencia": int(df["flag_sem_agencia"].sum()),
    "distribuicao_estrato": df["estrato_populacional"].value_counts().to_dict(),
}
save_json(features_report, "features_engineering_report.json")

2026-08-26 14:48:47,266 - INFO - Parquet salvo em /home/hbarboza/projects/tech/mba/heluvina_projeto_final_202602/data/processed/trusted_municipios_eda.parquet


2026-08-26 14:48:47,273 - INFO - JSON salvo em /home/hbarboza/projects/tech/mba/heluvina_projeto_final_202602/data/processed/reports/features_engineering_report.json


PosixPath('/home/hbarboza/projects/tech/mba/heluvina_projeto_final_202602/data/processed/reports/features_engineering_report.json')

## Resumo Executivo

Este notebook gerou o relatório `features_engineering_report.json`. Abaixo segue um resumo legível dos principais resultados.

In [9]:
# Resumo legível do relatório gerado
import json
from pathlib import Path

report_path = Path.cwd().parents[1] / "data/processed/reports/features_engineering_report.json"
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        report = json.load(f)
    print(json.dumps(report, indent=2, ensure_ascii=False))
else:
    print("Relatório ainda não foi gerado.")

{
  "features_adicionadas": [
    "pix_pj_pct",
    "pix_ticket_medio",
    "flag_sem_agencia",
    "estrato_populacional",
    "depositos_por_agencia",
    "credito_por_agencia"
  ],
  "nulos_pix_pj_pct": 0.0,
  "nulos_pix_ticket_medio": 0.0,
  "municipios_sem_agencia": 2656,
  "distribuicao_estrato": {
    "pequena": 4913,
    "media": 616,
    "grande": 41
  }
}
